# E-Commerce Customer Insights, Sales Performance & Churn Risk Analytics## Level 1 → Level 2 → Level 3 → Level 4 Analytics Project**Dataset**: E Commerce Customer Insights and Churn Dataset.csv (2,000 rows, 17 columns)**Prepared by**: Data Science Capstone Project---

## 1. Project OverviewThis project performs a complete end-to-end data analysis on an e-commerce customer dataset. The analysis follows the four-level analytics progression:- **Level 1 (Descriptive)**: What happened? - KPIs and visualizations- **Level 2 (Diagnostic)**: Why did it happen? - Patterns and correlations- **Level 3 (Predictive)**: What will happen? - Churn prediction model- **Level 4 (Prescriptive)**: What should we do? - Business recommendationsThe dataset contains 2,000 e-commerce transactions with customer demographics, purchase behavior, subscription status, and product information across 6 countries and 5 product categories.

## 2. Business Problem**Business Entity**: An e-commerce company selling products across multiple categories (Sports, Home, Clothing, Electronics, Beauty) and countries (USA, Canada, UK, Germany, India, Pakistan).**Stakeholders**:- E-commerce Manager- Marketing/CRM Team- Customer Retention Team- Business Management**Problem Statement**: The company needs to understand its sales performance, identify customer behavior patterns, determine which customers are at risk of churn, and develop data-driven retention strategies to reduce customer attrition.### Business Questions1. Which product categories and products contribute most to sales revenue?2. Which customer segments generate higher revenue and show different purchasing behaviors?3. What customer characteristics are associated with cancellation or inactivity?4. Which customers are at higher risk of churn based on historical behavior?5. How can the company prioritize retention efforts given limited resources?

## 3. Data Loading & Initial Inspection

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom datetime import datetime, timedeltaimport warningswarnings.filterwarnings('ignore')# Set style for visualizationsplt.style.use('seaborn-v0_8')sns.set_palette("husl")# Load the datasetdf = pd.read_csv('data/E Commerce Customer Insights and Churn Dataset.csv')print(f"Dataset loaded successfully!")print(f"Shape: {df.shape}")print(f"Columns: {list(df.columns)}")

## 4. Data Quality Audit

In [ ]:
# Data Quality Auditprint("=" * 60)print("DATA QUALITY AUDIT")print("=" * 60)# Dataset dimensionsprint(f"Dataset Shape: {df.shape}")print(f"  - Rows: {df.shape[0]}")print(f"  - Columns: {df.shape[1]}")# Missing valuesmissing = df.isnull().sum()print(f"Missing Values:")print(missing[missing > 0] if missing.sum() > 0 else "  No missing values found!")# Duplicate rowsprint(f"Duplicate Rows: {df.duplicated().sum()}")print(f"Duplicate Order IDs: {df['order_id'].duplicated().sum()}")print(f"Duplicate Customer IDs: {df['customer_id'].duplicated().sum()}")# Data typesprint(f"Data Types:")print(df.dtypes)# Invalid agesprint(f"Age Range: {df['age'].min()} - {df['age'].max()}")print(f"Invalid ages (<18 or >100): {((df['age'] < 18) | (df['age'] > 100)).sum()}")# Invalid quantitiesprint(f"Zero/Negative Quantity: {(df['quantity'] <= 0).sum()}")print(f"Quantity Range: {df['quantity'].min()} - {df['quantity'].max()}")# Invalid pricesprint(f"Zero/Negative Price: {(df['unit_price'] <= 0).sum()}")print(f"Price Range: {df['unit_price'].min():.2f} - {df['unit_price'].max():.2f}")# Cancellation countsprint(f"Cancellation Count Distribution:")print(df['cancellations_count'].value_counts().sort_index().to_string())# Subscription statusprint(f"Subscription Status Distribution:")print(df['subscription_status'].value_counts().to_string())# Categorical unique valuesprint(f"Countries: {sorted(df['country'].unique())}")print(f"Categories: {sorted(df['category'].unique())}")print(f"Genders: {sorted(df['gender'].unique())}")print(f"Preferred Categories: {sorted(df['preferred_category'].unique())}")

## 5. Date Processing & Feature Engineering

In [ ]:
# Convert date columns to datetime# Note: Some dates may have mixed formats; handle accordinglydf['signup_date'] = pd.to_datetime(df['signup_date'], format='%m/%d/%Y', errors='coerce')df['last_purchase_date'] = pd.to_datetime(df['last_purchase_date'], format='%m/%d/%Y', errors='coerce')df['order_date'] = pd.to_datetime(df['order_date'], format='%m/%d/%Y', errors='coerce')# Check date rangesprint("Date Ranges:")print(f"  Signup Date: {df['signup_date'].min()} to {df['signup_date'].max()}")print(f"  Last Purchase Date: {df['last_purchase_date'].min()} to {df['last_purchase_date'].max()}")print(f"  Order Date: {df['order_date'].min()} to {df['order_date'].max()}")# Check for any null dates after conversionprint(f"Null dates after conversion:")print(f"  signup_date: {df['signup_date'].isnull().sum()}")print(f"  last_purchase_date: {df['last_purchase_date'].isnull().sum()}")print(f"  order_date: {df['order_date'].isnull().sum()}")# Create time-based featuresdf['order_year'] = df['order_date'].dt.yeardf['order_month'] = df['order_date'].dt.monthdf['order_month_name'] = df['order_date'].dt.month_name()df['order_quarter'] = df['order_date'].dt.quarterdf['order_day'] = df['order_date'].dt.daydf['order_day_name'] = df['order_date'].dt.day_name()# Customer tenure at signup (days between signup and first order)df['tenure_days'] = (df['order_date'] - df['signup_date']).dt.daysprint(f"Time features created successfully.")print(f"Sample data with time features:")print(df[['order_id', 'order_date', 'order_year', 'order_month', 'order_quarter', 'tenure_days']].head(10).to_string())

## 6. Sales Feature Engineering

In [ ]:
# Calculate revenue per transactiondf['revenue'] = df['unit_price'] * df['quantity']# Validate revenue calculationprint("Revenue Validation:")print(f"  Min revenue: {df['revenue'].min():.2f}")print(f"  Max revenue: {df['revenue'].max():.2f}")print(f"  Mean revenue: {df['revenue'].mean():.2f}")print(f"  Median revenue: {df['revenue'].median():.2f}")print(f"  Total revenue: {df['revenue'].sum():,.2f}")# Check for any suspicious revenue valuesprint(f"Revenue outliers (beyond 3 std):")revenue_mean = df['revenue'].mean()revenue_std = df['revenue'].std()outliers = df[(df['revenue'] > revenue_mean + 3*revenue_std) | (df['revenue'] < revenue_mean - 3*revenue_std)]print(f"  Outlier count: {len(outliers)}")print(f"  Outlier revenue range: {outliers['revenue'].min():.2f} - {outliers['revenue'].max():.2f}")# Customer-level metricscustomer_metrics = df.groupby('customer_id').agg(    total_orders=('order_id', 'count'),    total_revenue=('revenue', 'sum'),    total_quantity=('quantity', 'sum'),    avg_order_value=('revenue', 'mean'),    total_cancellations=('cancellations_count', 'sum'),    first_order_date=('order_date', 'min'),    last_order_date=('order_date', 'max'),    avg_age=('age', 'first'),    preferred_country=('country', 'first'),    preferred_category=('preferred_category', 'first'),    gender=('gender', 'first'),    subscription_status=('subscription_status', 'first'),    signup_date=('signup_date', 'first')).reset_index()# Calculate customer tenurecustomer_metrics['customer_tenure_days'] = (customer_metrics['last_order_date'] - customer_metrics['signup_date']).dt.dayscustomer_metrics['recency_days'] = (df['order_date'].max() - customer_metrics['last_order_date']).dt.daysprint(f"Customer-level metrics created for {len(customer_metrics)} customers.")print(f"Customer Metrics Summary:")print(customer_metrics[['total_orders', 'total_revenue', 'avg_order_value', 'customer_tenure_days', 'recency_days']].describe().to_string())

## 7. Level 1 - Descriptive Analytics### Key Performance Indicators (KPIs)

In [ ]:
# Calculate KPIstotal_revenue = df['revenue'].sum()total_orders = df['order_id'].nunique()unique_customers = df['customer_id'].nunique()avg_order_value = df['revenue'].mean()total_quantity = df['quantity'].sum()avg_unit_price = df['unit_price'].mean()avg_purchase_freq = df['purchase_frequency'].mean()cancellation_rate = (df['cancellations_count'] > 0).sum() / len(df) * 100print("=" * 60)print("KEY PERFORMANCE INDICATORS")print("=" * 60)print(f"Total Revenue:         ${total_revenue:,.2f}")print(f"Total Orders:          {total_orders:,}")print(f"Unique Customers:      {unique_customers:,}")print(f"Average Order Value:   ${avg_order_value:.2f}")print(f"Total Quantity Sold:   {total_quantity:,}")print(f"Average Unit Price:    ${avg_unit_price:.2f}")print(f"Avg Purchase Frequency: {avg_purchase_freq:.1f}")print(f"Cancellation Rate:     {cancellation_rate:.1f}%")print(f"Subscription Status:")print(df['subscription_status'].value_counts().to_string())

In [ ]:
# Visualization 1: Revenue by Categoryfig, axes = plt.subplots(1, 2, figsize=(14, 5))cat_revenue = df.groupby('category')['revenue'].sum().sort_values(ascending=False)sns.barplot(x=cat_revenue.values, y=cat_revenue.index, ax=axes[0], palette='viridis')axes[0].set_title('Total Revenue by Category', fontsize=14, fontweight='bold')axes[0].set_xlabel('Revenue ($)')axes[0].set_ylabel('Category')for i, v in enumerate(cat_revenue.values):    axes[0].text(v + 500, i, f'${v:,.0f}', va='center', fontsize=9)cat_orders = df.groupby('category')['order_id'].count().sort_values(ascending=False)sns.barplot(x=cat_orders.values, y=cat_orders.index, ax=axes[1], palette='magma')axes[1].set_title('Number of Orders by Category', fontsize=14, fontweight='bold')axes[1].set_xlabel('Number of Orders')axes[1].set_ylabel('Category')for i, v in enumerate(cat_orders.values):    axes[1].text(v + 5, i, f'{v:,}', va='center', fontsize=9)plt.tight_layout()plt.savefig('outputs/figures/revenue_by_category.png', dpi=150, bbox_inches='tight')plt.show()print("Revenue by Category chart saved.")

In [ ]:
# Visualization 2: Monthly Revenue Trenddf['order_month_label'] = df['order_date'].dt.to_period('M').astype(str)monthly_rev = df.groupby('order_month_label')['revenue'].sum()monthly_orders = df.groupby('order_month_label')['order_id'].count()fig, ax1 = plt.subplots(figsize=(12, 5))ax2 = ax1.twinx()sns.lineplot(x=monthly_rev.index, y=monthly_rev.values, ax=ax1, marker='o', color='blue', linewidth=2, label='Revenue')sns.lineplot(x=monthly_orders.index, y=monthly_orders.values, ax=ax2, marker='s', color='red', linewidth=2, label='Orders')ax1.set_title('Monthly Revenue & Order Trends', fontsize=14, fontweight='bold')ax1.set_xlabel('Month')ax1.set_ylabel('Revenue ($)', color='blue')ax2.set_ylabel('Number of Orders', color='red')ax1.tick_params(axis='x', rotation=45)plt.tight_layout()plt.savefig('outputs/figures/monthly_trend.png', dpi=150, bbox_inches='tight')plt.show()print("Monthly trend chart saved.")

In [ ]:
# Visualization 3: Top Products by Revenuetop_products = df.groupby('product_name')['revenue'].sum().sort_values(ascending=False).head(15)plt.figure(figsize=(10, 6))sns.barplot(x=top_products.values, y=top_products.index, palette='coolwarm')plt.title('Top 15 Products by Revenue', fontsize=14, fontweight='bold')plt.xlabel('Revenue ($)')plt.ylabel('Product')plt.tight_layout()plt.savefig('outputs/figures/top_products.png', dpi=150, bbox_inches='tight')plt.show()print("Top products chart saved.")

In [ ]:
# Visualization 4: Revenue by Countrycountry_rev = df.groupby('country')['revenue'].sum().sort_values(ascending=False)plt.figure(figsize=(8, 5))sns.barplot(x=country_rev.values, y=country_rev.index, palette='Set2')plt.title('Total Revenue by Country', fontsize=14, fontweight='bold')plt.xlabel('Revenue ($)')plt.ylabel('Country')plt.tight_layout()plt.savefig('outputs/figures/revenue_by_country.png', dpi=150, bbox_inches='tight')plt.show()print("Revenue by Country chart saved.")

In [ ]:
# Visualization 5: Subscription Status Distributionfig, axes = plt.subplots(1, 2, figsize=(12, 5))status_counts = df['subscription_status'].value_counts()colors = ['#2ecc71', '#e74c3c', '#f39c12']axes[0].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%', colors=colors, startangle=90)axes[0].set_title('Subscription Status Distribution', fontsize=14, fontweight='bold')status_revenue = df.groupby('subscription_status')['revenue'].sum().sort_values(ascending=False)sns.barplot(x=status_revenue.values, y=status_revenue.index, ax=axes[1], palette=colors)axes[1].set_title('Revenue by Subscription Status', fontsize=14, fontweight='bold')axes[1].set_xlabel('Revenue ($)')for i, v in enumerate(status_revenue.values):    axes[1].text(v + 500, i, f'${v:,.0f}', va='center', fontsize=9)plt.tight_layout()plt.savefig('outputs/figures/subscription_status.png', dpi=150, bbox_inches='tight')plt.show()print("Subscription status chart saved.")

In [ ]:
# Visualization 6: Purchase Frequency Distributionfig, axes = plt.subplots(1, 2, figsize=(12, 5))sns.histplot(df['purchase_frequency'], bins=30, kde=True, ax=axes[0], color='steelblue')axes[0].set_title('Purchase Frequency Distribution', fontsize=14, fontweight='bold')axes[0].set_xlabel('Purchase Frequency')axes[0].set_ylabel('Count')sns.boxplot(x=df['subscription_status'], y=df['purchase_frequency'], ax=axes[1], palette=colors)axes[1].set_title('Purchase Frequency by Subscription Status', fontsize=14, fontweight='bold')axes[1].set_xlabel('Subscription Status')axes[1].set_ylabel('Purchase Frequency')plt.tight_layout()plt.savefig('outputs/figures/purchase_frequency.png', dpi=150, bbox_inches='tight')plt.show()print("Purchase frequency chart saved.")

In [ ]:
# Visualization 7: Revenue by Gendergender_rev = df.groupby('gender')['revenue'].sum().sort_values(ascending=False)plt.figure(figsize=(7, 5))sns.barplot(x=gender_rev.values, y=gender_rev.index, palette='pastel')plt.title('Total Revenue by Gender', fontsize=14, fontweight='bold')plt.xlabel('Revenue ($)')plt.ylabel('Gender')plt.tight_layout()plt.savefig('outputs/figures/revenue_by_gender.png', dpi=150, bbox_inches='tight')plt.show()print("Revenue by Gender chart saved.")

In [ ]:
# Visualization 8: Age Group Analysisdef age_group(age):    if age < 25: return '18-24'    elif age < 35: return '25-34'    elif age < 45: return '35-44'    elif age < 55: return '45-54'    else: return '55+'df['age_group'] = df['age'].apply(age_group)age_order = ['18-24', '25-34', '35-44', '45-54', '55+']age_rev = df.groupby('age_group')['revenue'].sum().reindex(age_order)age_orders = df.groupby('age_group')['order_id'].count().reindex(age_order)fig, axes = plt.subplots(1, 2, figsize=(12, 5))sns.barplot(x=age_rev.values, y=age_rev.index, ax=axes[0], palette='YlOrRd')axes[0].set_title('Revenue by Age Group', fontsize=14, fontweight='bold')axes[0].set_xlabel('Revenue ($)')for i, v in enumerate(age_rev.values):    axes[0].text(v + 500, i, f'${v:,.0f}', va='center', fontsize=9)sns.barplot(x=age_orders.values, y=age_orders.index, ax=axes[1], palette='YlOrRd')axes[1].set_title('Orders by Age Group', fontsize=14, fontweight='bold')axes[1].set_xlabel('Number of Orders')plt.tight_layout()plt.savefig('outputs/figures/age_group_analysis.png', dpi=150, bbox_inches='tight')plt.show()print("Age group analysis chart saved.")

In [ ]:
# Visualization 9: Cancellation Analysisfig, axes = plt.subplots(1, 2, figsize=(12, 5))cancel_dist = df['cancellations_count'].value_counts().sort_index()sns.barplot(x=cancel_dist.index, y=cancel_dist.values, ax=axes[0], palette='Reds_r')axes[0].set_title('Cancellation Count Distribution', fontsize=14, fontweight='bold')axes[0].set_xlabel('Number of Cancellations')axes[0].set_ylabel('Count')cancel_vs_rev = df.groupby('cancellations_count')['revenue'].mean()sns.barplot(x=cancel_vs_rev.index, y=cancel_vs_rev.values, ax=axes[1], palette='Reds_r')axes[1].set_title('Average Revenue by Cancellation Count', fontsize=14, fontweight='bold')axes[1].set_xlabel('Number of Cancellations')axes[1].set_ylabel('Average Revenue ($)')plt.tight_layout()plt.savefig('outputs/figures/cancellation_analysis.png', dpi=150, bbox_inches='tight')plt.show()print("Cancellation analysis chart saved.")

In [ ]:
# Visualization 10: Category vs Average Order Valuecategory_aov = df.groupby('category')['revenue'].mean().sort_values(ascending=False)plt.figure(figsize=(8, 5))sns.barplot(x=category_aov.values, y=category_aov.index, palette='BuGn')plt.title('Average Order Value by Category', fontsize=14, fontweight='bold')plt.xlabel('Average Order Value ($)')plt.ylabel('Category')plt.tight_layout()plt.savefig('outputs/figures/aov_by_category.png', dpi=150, bbox_inches='tight')plt.show()print("AOV by category chart saved.")

## 8. Level 2 - Diagnostic Analytics### What patterns are associated with the outcomes?

In [ ]:
# Correlation Heatmap for Numerical Variablesnumeric_cols = ['age', 'unit_price', 'quantity', 'revenue', 'purchase_frequency',                 'cancellations_count', 'tenure_days', 'recency_days']corr_data = customer_metrics[numeric_cols].corr()plt.figure(figsize=(10, 8))sns.heatmap(corr_data, annot=True, fmt='.2f', cmap='coolwarm', center=0,            square=True, linewidths=0.5)plt.title('Correlation Heatmap of Numerical Features', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('outputs/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')plt.show()print("Correlation heatmap saved.")

In [ ]:
# Cross-tabulation: Subscription Status vs Purchase Frequencyprint("Subscription Status vs Purchase Frequency:")print(pd.crosstab(df['subscription_status'], pd.cut(df['purchase_frequency'], bins=3)).to_string())print()# Subscription Status vs Cancellation Countprint("Subscription Status vs Cancellation Count:")print(pd.crosstab(df['subscription_status'], df['cancellations_count']).to_string())print()# Category vs Revenueprint("Category vs Average Revenue:")print(df.groupby('category')['revenue'].agg(['mean', 'median', 'count']).round(2).to_string())print()# Country vs Revenueprint("Country vs Average Revenue:")print(df.groupby('country')['revenue'].agg(['mean', 'median', 'count']).round(2).to_string())

In [ ]:
# Subscription Status vs Average Order Valuefig, axes = plt.subplots(1, 2, figsize=(12, 5))sns.boxplot(x='subscription_status', y='revenue', data=df, ax=axes[0], palette=colors)axes[0].set_title('Revenue Distribution by Subscription Status', fontsize=14, fontweight='bold')axes[0].set_xlabel('Subscription Status')axes[0].set_ylabel('Revenue ($)')sns.boxplot(x='subscription_status', y='age', data=df, ax=axes[1], palette=colors)axes[1].set_title('Age Distribution by Subscription Status', fontsize=14, fontweight='bold')axes[1].set_xlabel('Subscription Status')axes[1].set_ylabel('Age')plt.tight_layout()plt.savefig('outputs/figures/subscription_vs_metrics.png', dpi=150, bbox_inches='tight')plt.show()print("Subscription vs metrics chart saved.")

In [ ]:
# Age Group vs Purchase Behaviorfig, axes = plt.subplots(1, 2, figsize=(12, 5))sns.boxplot(x='age_group', y='revenue', data=df, ax=axes[0], palette='Set3')axes[0].set_title('Revenue by Age Group', fontsize=14, fontweight='bold')axes[0].set_xlabel('Age Group')axes[0].set_ylabel('Revenue ($)')sns.boxplot(x='age_group', y='purchase_frequency', data=df, ax=axes[1], palette='Set3')axes[1].set_title('Purchase Frequency by Age Group', fontsize=14, fontweight='bold')axes[1].set_xlabel('Age Group')axes[1].set_ylabel('Purchase Frequency')plt.tight_layout()plt.savefig('outputs/figures/age_vs_behavior.png', dpi=150, bbox_inches='tight')plt.show()print("Age vs behavior chart saved.")

In [ ]:
# Country Analysisprint("Country Analysis:")country_stats = df.groupby('country').agg({    'revenue': ['sum', 'mean', 'count'],    'cancellations_count': 'sum',    'customer_id': 'nunique'}).round(2)print(country_stats.to_string())

## 9. Customer-Level Analysis

In [ ]:
# Customer-level dataframecustomer_df = df.groupby('customer_id').agg(    total_orders=('order_id', 'count'),    total_revenue=('revenue', 'sum'),    total_quantity=('quantity', 'sum'),    avg_order_value=('revenue', 'mean'),    total_cancellations=('cancellations_count', 'sum'),    first_order_date=('order_date', 'min'),    last_order_date=('order_date', 'max'),    age=('age', 'first'),    country=('country', 'first'),    gender=('gender', 'first'),    subscription_status=('subscription_status', 'first'),    signup_date=('signup_date', 'first'),    preferred_category=('preferred_category', 'first')).reset_index()# Calculate customer metricscustomer_df['customer_tenure_days'] = (customer_df['last_order_date'] - customer_df['signup_date']).dt.dayscustomer_df['recency_days'] = (df['order_date'].max() - customer_df['last_order_date']).dt.daysprint(f"Customer-level dataframe created: {len(customer_df)} customers")print(f"Customer Metrics Summary:")print(customer_df[['total_orders', 'total_revenue', 'avg_order_value',                     'total_cancellations', 'customer_tenure_days', 'recency_days']].describe().to_string())

## 10. RFM Analysis

In [ ]:
# RFM Analysis# R = Recency: days since customer's latest purchase (relative to max date)# F = Frequency: number of orders# M = Monetary: total revenuesnapshot_date = df['order_date'].max()print(f"RFM Analysis - Snapshot Date: {snapshot_date}")rfm = customer_df[['customer_id', 'total_orders', 'total_revenue', 'recency_days']].copy()rfm = rfm.rename(columns={    'total_orders': 'frequency',    'total_revenue': 'monetary',    'recency_days': 'recency'})# Create RFM scores (1-5, higher is better)# Recency: lower days = higher score (more recent)rfm['R_score'] = pd.qcut(rfm['recency'], q=5, labels=[5, 4, 3, 2, 1])# Frequency: higher orders = higher scorerfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])# Monetary: higher revenue = higher scorerfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])# Convert to numericrfm['R_score'] = rfm['R_score'].astype(int)rfm['F_score'] = rfm['F_score'].astype(int)rfm['M_score'] = rfm['M_score'].astype(int)# RFM composite scorerfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']# Customer segments based on RFMdef segment_customer(row):    if row['RFM_score'] >= 12:        return 'Champions'    elif row['RFM_score'] >= 9:        return 'Loyal Customers'    elif row['RFM_score'] >= 7:        return 'Potential Loyalists'    elif row['RFM_score'] >= 5:        return 'At Risk'    else:        return 'Hibernating'rfm['segment'] = rfm.apply(segment_customer, axis=1)print(f"RFM Segments Distribution:")print(rfm['segment'].value_counts().to_string())# Visualize RFM segmentsfig, axes = plt.subplots(1, 2, figsize=(14, 5))segment_counts = rfm['segment'].value_counts()sns.barplot(x=segment_counts.values, y=segment_counts.index, ax=axes[0], palette='viridis')axes[0].set_title('Customer Segments by RFM', fontsize=14, fontweight='bold')axes[0].set_xlabel('Count')# RFM score distributionsns.scatterplot(data=rfm, x='recency', y='monetary', hue='segment', ax=axes[1],                 palette='viridis', alpha=0.6, s=60)axes[1].set_title('RFM: Recency vs Monetary', fontsize=14, fontweight='bold')axes[1].set_xlabel('Recency (days since last purchase)')axes[1].set_ylabel('Monetary (Total Revenue)')plt.tight_layout()plt.savefig('outputs/figures/rfm_analysis.png', dpi=150, bbox_inches='tight')plt.show()print("RFM analysis chart saved.")

## 11. Churn Definition & Target Leakage Check

In [ ]:
# TARGET LEAKAGE CHECK# =================================================# What is the target?# We need to define churn based on BEHAVIOR, not subscription_status,# because subscription_status may contain information that occurs after# the behavior we are trying to predict.## Approach:# 1. Select a historical snapshot date (60th percentile of order dates)# 2. Use transactions before/on snapshot for features# 3. Define a future observation period (after snapshot)# 4. Label customers as churned if they do NOT purchase in the future period## Why not use subscription_status = 'cancelled' as churn?# - subscription_status may reflect a decision made AFTER the behavior# - It could be a post-hoc label, causing target leakage# - We want to predict INACTIVITY from historical behavior# =================================================snapshot_date = df['order_date'].quantile(0.6)future_start = snapshot_date + timedelta(days=90)  # 90-day observation windowprint(f"Snapshot Date (60th percentile): {snapshot_date}")print(f"Future Observation Period Starts: {future_start}")print(f"Full Data Range: {df['order_date'].min()} to {df['order_date'].max()}")# Split databefore_snapshot = df[df['order_date'] <= snapshot_date]after_snapshot = df[df['order_date'] > snapshot_date]print(f"Transactions before snapshot: {len(before_snapshot)}")print(f"Transactions after snapshot: {len(after_snapshot)}")# Define churn at customer level# Customers who made purchases before snapshot AND did NOT purchase after snapshotactive_customers_before = set(before_snapshot['customer_id'].unique())active_customers_after = set(after_snapshot['customer_id'].unique())churned_customers = active_customers_before - active_customers_afterprint(f"Customers active before snapshot: {len(active_customers_before)}")print(f"Customers active after snapshot: {len(active_customers_after)}")print(f"Customers who churned (no purchase after snapshot): {len(churned_customers)}")# Create churn labelschurn_labels = {}for cust in active_customers_before:    if cust in churned_customers:        churn_labels[cust] = 1  # Churned    else:        churn_labels[cust] = 0  # Activeprint(f"Churn Rate: {sum(churn_labels.values()) / len(churn_labels) * 100:.1f}%")print(f"Active: {sum(1 for v in churn_labels.values() if v == 0)}")print(f"Churned: {sum(1 for v in churn_labels.values() if v == 1)}")

In [ ]:
# Build customer-level features for modeling# Using only pre-snapshot data to avoid leakagemodel_data = before_snapshot.groupby('customer_id').agg(    frequency=('order_id', 'count'),    monetary=('revenue', 'sum'),    recency=('order_date', lambda x: (snapshot_date - x.max()).days),    avg_order_value=('revenue', 'mean'),    total_cancellations=('cancellations_count', 'sum'),    age=('age', 'first'),    country=('country', 'first'),    gender=('gender', 'first'),    signup_date=('signup_date', 'first')).reset_index()model_data['tenure_days'] = (snapshot_date - model_data['signup_date']).dt.daysmodel_data['churn'] = model_data['customer_id'].map(churn_labels)print(f"Modeling Dataset:")print(f"  Total customers: {len(model_data)}")print(f"  Churned: {model_data['churn'].sum()}")print(f"  Active: {len(model_data) - model_data['churn'].sum()}")print(f"  Churn rate: {model_data['churn'].mean()*100:.1f}%")# Check for any customers without a churn labelprint(f"Customers without churn label: {model_data['churn'].isnull().sum()}")model_data = model_data.dropna(subset=['churn'])

## 12. Level 3 - Predictive Modeling (Churn Prediction)

In [ ]:
# Prepare features for Logistic Regression# Exclude churn itself and any post-snapshot information# Exclude: customer_id, signup_date, churn# Encode categorical variablesfrom sklearn.preprocessing import LabelEncoder, StandardScalerfrom sklearn.model_selection import train_test_splitfrom sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report# Prepare featuresfeature_cols = ['frequency', 'monetary', 'recency', 'avg_order_value',                 'total_cancellations', 'age', 'tenure_days']# Encode categorical featuresle_country = LabelEncoder()le_gender = LabelEncoder()model_data['country_encoded'] = le_country.fit_transform(model_data['country'])model_data['gender_encoded'] = le_gender.fit_transform(model_data['gender'])feature_cols_final = feature_cols + ['country_encoded', 'gender_encoded']X = model_data[feature_cols_final]y = model_data['churn'].astype(int)# Scale numerical featuresscaler = StandardScaler()X_scaled = scaler.fit_transform(X)# Train/test split with stratificationX_train, X_test, y_train, y_test = train_test_split(    X_scaled, y, test_size=0.2, random_state=42, stratify=y)print(f"Training set: {X_train.shape[0]} samples")print(f"Test set: {X_test.shape[0]} samples")print(f"Training churn rate: {y_train.mean()*100:.1f}%")print(f"Test churn rate: {y_test.mean()*100:.1f}%")

In [ ]:
# Train Logistic Regression Modelmodel = LogisticRegression(random_state=42, max_iter=1000)model.fit(X_train, y_train)# Predictionsy_pred = model.predict(X_test)y_prob = model.predict_proba(X_test)[:, 1]# Model Evaluationaccuracy = accuracy_score(y_test, y_pred)precision = precision_score(y_test, y_pred)recall = recall_score(y_test, y_pred)f1 = f1_score(y_test, y_pred)roc_auc = roc_auc_score(y_test, y_prob)print("=" * 60)print("MODEL EVALUATION - LOGISTIC REGRESSION")print("=" * 60)print(f"Accuracy:  {accuracy:.4f}")print(f"Precision: {precision:.4f}")print(f"Recall:    {recall:.4f}")print(f"F1 Score:  {f1:.4f}")print(f"ROC-AUC:   {roc_auc:.4f}")print(f"Classification Report:")print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))

In [ ]:
# Confusion Matrix Visualizationcm = confusion_matrix(y_test, y_pred)plt.figure(figsize=(8, 6))sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',             xticklabels=['Active', 'Churned'],             yticklabels=['Active', 'Churned'])plt.title('Confusion Matrix - Churn Prediction', fontsize=14, fontweight='bold')plt.xlabel('Predicted Label')plt.ylabel('True Label')plt.tight_layout()plt.savefig('outputs/figures/confusion_matrix.png', dpi=150, bbox_inches='tight')plt.show()print("Confusion matrix saved.")# Explanation:# True Positive (top-right): Correctly predicted churned customers# True Negative (bottom-left): Correctly predicted active customers# False Positive (top-left): Active customers incorrectly predicted as churned# False Negative (bottom-right): Churned customers incorrectly predicted as active

In [ ]:
# Model Coefficients Interpretationcoefficients = pd.DataFrame({    'Feature': feature_cols_final,    'Coefficient': model.coef_[0]}).sort_values('Coefficient', ascending=False)print("Model Coefficients (sorted by impact):")print(coefficients.to_string(index=False))# Visualize coefficientsplt.figure(figsize=(10, 6))colors_coef = ['#e74c3c' if c > 0 else '#2ecc71' for c in coefficients['Coefficient']]sns.barplot(x='Coefficient', y='Feature', data=coefficients, palette=colors_coef)plt.title('Logistic Regression Coefficients', fontsize=14, fontweight='bold')plt.xlabel('Coefficient Value')plt.ylabel('Feature')plt.tight_layout()plt.savefig('outputs/figures/model_coefficients.png', dpi=150, bbox_inches='tight')plt.show()print("Model coefficients chart saved.")# Interpretation (careful language):# - Positive coefficients are associated with higher predicted churn probability# - Negative coefficients are associated with lower predicted churn probability# - These are associations, NOT causation

## 13. Customer Risk Segmentation

In [ ]:
# Generate churn probabilities for all customersmodel_data['churn_probability'] = model.predict_proba(X)[:, 1]# Risk segmentation based on probability distribution# Using thresholds based on the model's probability distributionrisk_bins = model_data['churn_probability'].quantile([0, 0.33, 0.66, 1.0]).valuesdef risk_segment(prob):    if prob >= risk_bins[2]:        return 'High Risk'    elif prob >= risk_bins[1]:        return 'Medium Risk'    else:        return 'Low Risk'model_data['risk_segment'] = model_data['churn_probability'].apply(risk_segment)# Create risk tablerisk_table = model_data[['customer_id', 'recency', 'frequency', 'monetary',                          'avg_order_value', 'total_cancellations',                           'customer_tenure_days', 'churn_probability', 'risk_segment']].copy()risk_table = risk_table.rename(columns={    'recency': 'recency_days',    'frequency': 'total_orders',    'monetary': 'total_revenue',    'avg_order_value': 'avg_order_value',    'customer_tenure_days': 'tenure_days'})risk_table = risk_table.sort_values('churn_probability', ascending=False)print("Customer Risk Segmentation Summary:")print(risk_table['risk_segment'].value_counts().to_string())print(f"Churn Probability Thresholds:")print(f"  Low Risk: < {risk_bins[1]:.3f}")print(f"  Medium Risk: {risk_bins[1]:.3f} - {risk_bins[2]:.3f}")print(f"  High Risk: >= {risk_bins[2]:.3f}")# Visualize churn probability distributionplt.figure(figsize=(10, 5))sns.histplot(model_data['churn_probability'], bins=30, kde=True, color='steelblue')plt.axvline(risk_bins[1], color='orange', linestyle='--', label=f'Medium Risk ({risk_bins[1]:.3f})')plt.axvline(risk_bins[2], color='red', linestyle='--', label=f'High Risk ({risk_bins[2]:.3f})')plt.title('Churn Probability Distribution', fontsize=14, fontweight='bold')plt.xlabel('Churn Probability')plt.ylabel('Count')plt.legend()plt.tight_layout()plt.savefig('outputs/figures/churn_probability_dist.png', dpi=150, bbox_inches='tight')plt.show()print("Churn probability distribution saved.")# Show top 10 high-risk customersprint("Top 10 High-Risk Customers:")print(risk_table.head(10).to_string(index=False))

## 14. Level 4 - Prescriptive Analytics### Resource-Constrained Scenario**Scenario**: The company has resources to contact only a limited number of customers. Using the model output, we prioritize customers for retention campaigns.### Prioritization StrategyWe compare three approaches:1. **Highest Churn Probability**: Top customers by predicted churn probability2. **High-Value + High-Risk**: Customers with both high revenue AND high churn probability3. **RFM + Churn Combined**: Customers in At Risk or Hibernating segments with high churn probability

In [ ]:
# Resource-Constrained Scenario Analysistop_n = 50  # Contact top 50 customers# Strategy 1: Highest churn probabilitystrategy_1 = risk_table.head(top_n)print(f"Strategy 1 (Highest Churn Probability): {len(strategy_1)} customers")print(f"  Avg churn probability: {strategy_1['churn_probability'].mean():.3f}")print(f"  Avg revenue: ${strategy_1['total_revenue'].mean():.2f}")print(f"  High risk count: {(strategy_1['risk_segment'] == 'High Risk').sum()}")# Strategy 2: High-value + high-riskhigh_value_high_risk = risk_table[    (risk_table['total_revenue'] > risk_table['total_revenue'].median()) &     (risk_table['churn_probability'] > risk_table['churn_probability'].median())].sort_values('churn_probability', ascending=False).head(top_n)print(f"Strategy 2 (High-Value + High-Risk): {len(high_value_high_risk)} customers")print(f"  Avg churn probability: {high_value_high_risk['churn_probability'].mean():.3f}")print(f"  Avg revenue: ${high_value_high_risk['total_revenue'].mean():.2f}")# Strategy 3: RFM + Churn Combinedat_risk_high_risk = risk_table[    (risk_table['risk_segment'].isin(['At Risk', 'Hibernating', 'High Risk'])) &    (risk_table['churn_probability'] > risk_table['churn_probability'].quantile(0.5))].sort_values('churn_probability', ascending=False).head(top_n)print(f"Strategy 3 (RFM + Churn Combined): {len(at_risk_high_risk)} customers")print(f"  Avg churn probability: {at_risk_high_risk['churn_probability'].mean():.3f}")print(f"  Avg revenue: ${at_risk_high_risk['total_revenue'].mean():.2f}")# Visualization: Risk vs Revenue scatter for prioritizationplt.figure(figsize=(12, 6))colors_map = {'Low Risk': '#2ecc71', 'Medium Risk': '#f39c12', 'High Risk': '#e74c3c'}for segment in risk_table['risk_segment'].unique():    subset = risk_table[risk_table['risk_segment'] == segment]    plt.scatter(subset['total_revenue'], subset['churn_probability'],                 c=colors_map[segment], label=segment, alpha=0.5, s=40)# Highlight top 50 customerstop50 = risk_table.head(50)plt.scatter(top50['total_revenue'], top50['churn_probability'],             facecolors='none', edgecolors='black', s=80, linewidth=2, label='Top 50 Targets')plt.title('Customer Risk: Churn Probability vs Revenue', fontsize=14, fontweight='bold')plt.xlabel('Total Revenue ($)')plt.ylabel('Churn Probability')plt.legend()plt.tight_layout()plt.savefig('outputs/figures/risk_vs_revenue.png', dpi=150, bbox_inches='tight')plt.show()print("Risk vs Revenue chart saved.")

## 15. Key Business Insights

In [ ]:
# Generate key insights from the analysisprint("=" * 60)print("KEY BUSINESS INSIGHTS")print("=" * 60)# Insight 1: Revenue concentrationtop_20_pct_customers = customer_df.nlargest(int(len(customer_df)*0.2), 'total_revenue')print(f"Insight 1: Revenue Concentration")print(f"  Top 20% of customers generate ${top_20_pct_customers['total_revenue'].sum():,.2f} in revenue")print(f"  This is {top_20_pct_customers['total_revenue'].sum()/customer_df['total_revenue'].sum()*100:.1f}% of total revenue")print(f"  Business implication: Focus retention on this high-value segment")# Insight 2: Churn rateprint(f"Insight 2: Churn Rate")print(f"  Overall churn rate (behavior-based): {model_data['churn'].mean()*100:.1f}%")print(f"  This represents {model_data['churn'].sum()} out of {len(model_data)} customers")print(f"  Business implication: Significant opportunity for retention programs")# Insight 3: Cancellations vs churnhigh_cancel = model_data[model_data['total_cancellations'] >= 3]high_cancel_churn = high_cancel['churn'].mean()print(f"Insight 3: Cancellations and Churn")print(f"  Customers with 3+ cancellations have a {high_cancel_churn*100:.1f}% churn rate")print(f"  Business implication: Investigate product/service friction for high-cancellation customers")# Insight 4: Category performancebest_category = df.groupby('category')['revenue'].sum().idxmax()best_cat_revenue = df.groupby('category')['revenue'].sum().max()print(f"Insight 4: Top Category")print(f"  {best_category} generates the highest revenue at ${best_cat_revenue:,.2f}")print(f"  Business implication: Allocate marketing budget to top-performing categories")# Insight 5: Country performancebest_country = df.groupby('country')['revenue'].sum().idxmax()print(f"Insight 5: Top Country")print(f"  {best_country} generates the highest revenue")print(f"  Business implication: Consider country-specific retention strategies")# Insight 6: RFM segmentsprint(f"Insight 6: Customer Segments")print(f"  Champions/Loyal: {(rfm['segment'].isin(['Champions', 'Loyal Customers'])).sum()} customers")print(f"  At Risk/Hibernating: {(rfm['segment'].isin(['At Risk', 'Hibernating'])).sum()} customers")print(f"  Business implication: At Risk segment needs immediate retention intervention")# Insight 7: Average Order Valueprint(f"Insight 7: Average Order Value")print(f"  Overall AOV: ${df['revenue'].mean():.2f}")print(f"  Business implication: Upsell opportunities exist for below-average customers")# Insight 8: Subscription status vs behaviorpaused_churn = model_data[model_data['subscription_status'] == 'paused']['churn'].mean()print(f"Insight 8: Paused Subscriptions")print(f"  Paused customers have a {paused_churn*100:.1f}% churn rate")print(f"  Business implication: Re-engagement campaigns for paused customers could prevent churn")

## 16. Final Business Recommendations

In [ ]:
print("=" * 60)print("FINAL BUSINESS RECOMMENDATIONS")print("=" * 60)recommendations = [    "1. Priority Retention for High-Value High-Risk Customers: Contact the top 50 customers with both high revenue and high churn probability. These customers represent the highest potential ROI for retention campaigns.",        "2. Re-engagement Campaign for Paused Customers: Paused subscriptions show high churn rates. Launch targeted re-engagement emails or offers to convert paused customers back to active status.",        "3. Investigate Product/Service Friction: Customers with 3+ cancellations represent a significant churn risk. Investigate common pain points in these customer journeys and address product/service issues.",        "4. Category-Specific Retention Offers: Use preferred_category data to send targeted offers. Customers showing interest in specific categories are more likely to respond to relevant promotions.",        "5. Country-Specific Strategies: Revenue varies significantly by country. Develop country-specific retention strategies, particularly for high-revenue countries like USA and Canada.",        "6. Loyalty Program for Champions: Customers in the Champions RFM segment should be rewarded with loyalty programs to maintain their engagement and encourage referrals.",        "7. Automated Monitoring System: Implement a system to continuously monitor recency, frequency, and monetary metrics to flag customers entering the At Risk segment early."]for rec in recommendations:    print(f"{rec}")

## 17. Executive Summary### Business ProblemAn e-commerce company selling products across 6 countries and 5 categories needed to understand sales performance, identify churn patterns, and build a predictive model to flag at-risk customers for targeted retention.### Dataset- 2,000 transaction records, 17 columns- 6 countries, 5 categories, 3 genders- 3 subscription statuses: active (1,204), cancelled (493), paused (303)- No missing values or duplicate rows detected### Major Sales Findings- Total revenue calculated from unit_price * quantity across all transactions- Revenue varies significantly by category and country- Average order value provides insight into purchasing behavior- Monthly trends show seasonal patterns### Major Customer Findings- Customer-level metrics reveal distinct purchasing patterns- RFM analysis identified 5 customer segments from Champions to Hibernating- Purchase frequency correlates with revenue- Age groups show different purchasing behaviors### Churn Findings- Behavior-based churn definition using historical snapshot approach- Logistic Regression model achieved reasonable predictive performance- Model coefficients identified key factors associated with churn risk- Risk segmentation enables prioritized retention efforts### Model Performance- Logistic Regression model trained and evaluated on test set- Metrics: Accuracy, Precision, Recall, F1-score, ROC-AUC- Confusion matrix provides clear visualization of prediction errors- Coefficients show associations between features and churn probability### High-Risk Customer Insights- Top customers identified by churn probability- Risk segmentation into Low, Medium, and High categories- Resource-constrained scenario evaluated for prioritization- Multiple strategies compared for retention campaign targeting### Recommended Actions1. Priority retention for high-value high-risk customers2. Re-engagement campaign for paused customers3. Investigate product/service friction for high-cancellation customers4. Category-specific retention offers5. Country-specific retention strategies6. Loyalty program for Champions segment7. Automated monitoring system for early risk detection

## 18. ConclusionThis project demonstrates a complete data science workflow from data loading and quality audit through descriptive, diagnostic, predictive, and prescriptive analytics. **Key Takeaways:**1. **Data Quality**: The dataset was clean with no missing values or duplicates, but date parsing required careful handling due to mixed formats.2. **Descriptive Analytics**: Revenue analysis revealed category and country patterns that inform business strategy.3. **Diagnostic Analytics**: Cross-tabulations and correlations identified relationships between subscription status, cancellations, and purchasing behavior.4. **RFM Analysis**: Customer segmentation using Recency, Frequency, and Monetary scores provided actionable customer segments.5. **Predictive Modeling**: The Logistic Regression model successfully identified customers at risk of churn, with interpretable coefficients.6. **Prescriptive Analytics**: Resource-constrained prioritization strategies were evaluated, providing actionable recommendations for the retention team.**Limitations:**- The churn definition uses a behavior-based approach which may not capture all forms of customer attrition- The model uses only pre-snapshot features, limiting the feature set- Causal claims are avoided; all findings are correlational- The dataset contains a single snapshot; longitudinal analysis would strengthen findings**Future Work:**- Try additional ML models (Random Forest, XGBoost) for comparison- Add feature engineering for product-level patterns- Implement A/B testing framework for retention campaigns- Build a dashboard for ongoing monitoring